# MCXE31 Debug Authentication

This Jupyter notebook demonstrates how to perform SDA authentication on MCXE31 devices using SPSDK. The notebook covers both Password Mode and Challenge/Response Mode authentication methods.

## 1. Introduction <a name="introduction"></a>

The MCXE31 device uses HSE (Hardware Security Engine) for secure debug access control. Debug authentication is based on a 128-bit secret called ADKP (Application Debug Key/Password).

### Authentication Methods

**Static (Password Mode):**
- ADKP is used as a password provided in plain form
- Less secure but simpler
- Debug authentication mode: `PASSWORD (0x0)`

**Dynamic (Challenge/Response Mode - Recommended):**
- ADKP is a cryptographic key used to calculate a response to a random challenge
- More secure as secrets are never exposed in plaintext
- Debug authentication mode: `CHALLENGE_RESPONSE (0x1)`

### ADKP Diversification (Optional but Recommended)

ADKP can be diversified with the device's UID to create device-specific debug keys:
- Host provisions a master ADKP
- HSE internally derives: `actual_ADKP = derive(master_ADKP, UID)`
- For authentication, the host must derive the same device-specific key

---
## 2. Prerequisites <a name="prerequisites"></a>

### Requirements
- MCXE31 development board
- Debug probe (J-Link, PE Micro, etc.)
- USB cable for debug probe connection
- SPSDK installed (`pip install spsdk`)

### Device Provisioning Requirements

Before performing debug authentication, the device must be provisioned:

1. **ADKP must be set** using `blhost key-provisioning set_user_key 0` attribute
2. **Lifecycle must be advanced** to OEM_PROD or IN_FIELD using `nxpele hse set-attr secure-lifecycle`

**Note:** These provisioning steps are typically done once during device manufacturing/provisioning.

---
## 3. Setup and Configuration <a name="setup"></a>

In [1]:
# Import required modules

# SPSDK imports
from spsdk.dat.sda import SdaAuthentication
from spsdk.utils.family import FamilyRevision

# This env variable sets colored logger output to STDOUT
%env JUPYTER_SPSDK=1
# Set a magic for command execution and echo
%alias execute echo %l && %l
%alias_magic ! execute

FAMILY = "mcxe31b"  # Device family
REVISION = "latest"  # Device revision
DEBUG_INTERFACE = "jlink"  # Debug probe interface (jlink, pemicro, pyocd, etc.)
PORT = "-p COM21"
KEYS = "keys/"
ADKP_KEY = KEYS + "adkp.bin"  # ADKP key for provisioning

env: JUPYTER_SPSDK=1
Created `%!` as an alias for `%execute`.


In [2]:
# Verify SDA is supported for this family
supported_families = SdaAuthentication.get_supported_families()
print(f"SDA Supported Families: {[f.name for f in supported_families]}")

family = FamilyRevision(FAMILY)
if family not in supported_families:
    print(f"⚠️ WARNING: {family.name} may not be in the supported families list")
else:
    print(f"✓ {family.name} supports SDA authentication")

SDA Supported Families: ['mcxe315', 'mcxe316', 'mcxe317', 'mcxe31b']
✓ mcxe31b supports SDA authentication


---
## 4. Setting Debug Authentication Mode <a name="set-auth-mode"></a>

Before performing debug authentication, you must configure the debug authentication mode on the device. This is done using the `nxpele hse set-attr debug-auth-mode` command.

### Important Notes:
- This step must be performed **before** advancing the lifecycle to OEM_PROD or IN_FIELD
- This is a **one-time programmable** attribute (can only be set once)
- Requires device to be in CUST_DEL lifecycle
- Requires ADKP to be provisioned first

### 4.1 Set Debug Authentication Mode via CLI

Use the following commands to set the debug authentication mode. The default authentication mode is password. In order to change it to challenge-response(recommended), run following command:

In [3]:
%! nxpele $PORT -f $FAMILY hse set-attr debug-auth-mode -v challenge-response

nxpele -p COM21 -f mcxe31b hse set-attr debug-auth-mode -v challenge-response 
Setting up the attribute 'DEBUG_AUTH_MODE' finished: Succeeded.


### 4.2 Verify Debug Authentication Mode

You can verify the current debug authentication mode using:

In [4]:
%! nxpele $PORT -f $FAMILY hse get-attr -id debug_auth_mode

nxpele -p COM21 -f mcxe31b hse get-attr -id debug_auth_mode 
HSE Debug Authentication Mode:
  Mode: challenge-response (0x01)
  Description: Challenge-response based debug authorization (dynamic)

Attribute Properties:
  - One-Time Programmable (OTP) - can only be written once
  - Determines authentication method for debug access
  - PASSWORD: Uses static token/password authentication
  - CHALLENGE_RESPONSE: Uses dynamic challenge-response authentication



---
## 5. Password Mode Authentication <a name="password-mode"></a>

Password mode uses the ADKP as a static password. The debugger transmits the 128-bit password to the device.

### Password Mode Flow:
1. Debugger transmits 128-bit password to KEYRESPn registers (bits 0-127)
2. Remaining bits (128-255) are set to zero (padding)
3. If password matches ADKP, debug connection is opened
4. Otherwise, debug connection remains closed

### Security Note:
Password mode is less secure than Challenge/Response mode because the password is transmitted in plaintext.

### 5.1 Password Mode via CLI

You can also perform password mode authentication using the CLI:

In [5]:
%! nxpdebugmbox -i jlink sda auth -f $FAMILY -t password -a $ADKP_KEY

nxpdebugmbox -i jlink sda auth -f mcxe31b -t password -a keys/adkp.bin 
  #   Interface   Id          Description         
--------------------------------------------------
  0   Jlink       602010037   Segger J-Link PLUS  
The SDA DebugAuthMode.PASSWORD authentication succeeded.


---
## 6. Challenge/Response Mode Authentication <a name="challenge-response-mode"></a>

Challenge/Response mode is the recommended authentication method. It uses cryptographic operations to authenticate without exposing secrets in plaintext.

### Challenge/Response Flow:
1. Debugger reads 256-bit challenge from KEYCHALn registers
2. Debugger computes response using AES-128-ECB encryption:
   - Split challenge into two 128-bit blocks
   - Encrypt each block with ADKP key
   - Concatenate encrypted blocks to form 256-bit response
3. Debugger writes response to KEYRESPn registers
4. If response matches HSE's internal calculation, debug connection is opened

### Security Advantages:
- ADKP key is never transmitted in plaintext
- Each authentication uses a unique random challenge
- Resistant to replay attacks

### 6.1 Challenge/Response Mode via CLI

You can also perform challenge/response mode authentication using the CLI:

In [6]:
%! nxpdebugmbox -i jlink sda auth -f $FAMILY -t challenge-response -a $ADKP_KEY

nxpdebugmbox -i jlink sda auth -f mcxe31b -t challenge-response -a keys/adkp.bin 
  #   Interface   Id          Description         
--------------------------------------------------
  0   Jlink       602010037   Segger J-Link PLUS  
The SDA DebugAuthMode.CHALLENGE_RESPONSE authentication succeeded.
